In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

In [3]:
# !pip install openexr Imath numpy matplotlib
!pip install opencv-python-headless

In [4]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import cv2
import matplotlib.pyplot as plt

def read_exr(path):
    img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    img = img.astype(np.float32)

    # BGR → RGB
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # HWC → CHW
    img = np.transpose(img, (2, 0, 1))

    return img

In [5]:
img = cv2.imread("/content/drive/MyDrive/469_data/room3/25136394-08192spp.exr", cv2.IMREAD_UNCHANGED)
print(img is None)

False


In [6]:
class MonteCarloDataset(Dataset):
    def __init__(self, root_dirs, patch_size=128):
        self.pairs = []
        self.patch_size = patch_size

        for root in root_dirs:
            files = os.listdir(root)

            for f in files:
                if "00128spp.exr" in f:
                    base = f.split("-00128spp.exr")[0]
                    clean_name = base + "-08192spp.exr"

                    noisy_path = os.path.join(root, f)
                    clean_path = os.path.join(root, clean_name)

                    if os.path.exists(clean_path):
                        self.pairs.append((noisy_path, clean_path))

        print("Total pairs:", len(self.pairs))

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        noisy_path, clean_path = self.pairs[idx]

        noisy = read_exr(noisy_path)
        clean = read_exr(clean_path)

        C, H, W = noisy.shape
        ps = self.patch_size

        top = np.random.randint(0, H - ps)
        left = np.random.randint(0, W - ps)

        noisy = noisy[:, top:top+ps, left:left+ps]
        clean = clean[:, top:top+ps, left:left+ps]

        noisy = torch.from_numpy(noisy)
        clean = torch.from_numpy(clean)

        return noisy, clean

In [7]:
train_dataset = MonteCarloDataset([
    "/content/drive/MyDrive/469_data/room2",
    "/content/drive/MyDrive/469_data/room3"
])

Total pairs: 388


In [8]:
test_dataset = MonteCarloDataset([
    "/content/drive/MyDrive/469_data/car2"
])

Total pairs: 180


In [9]:
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=0)

In [10]:
from kpcn_model import KPCN

device = "cuda" if torch.cuda.is_available() else "cpu"

model = KPCN().to(device)

In [ ]:
# =========================
# TRAIN 10 EPOCHS
# =========================

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = torch.nn.L1Loss()

epochs = 10

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for noisy, clean in train_loader:
        noisy = noisy.to(device)
        clean = clean.to(device)

        optimizer.zero_grad()
        output = model(noisy)
        loss = criterion(output, clean)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}] Loss: {total_loss/len(train_loader):.4f}")

In [ ]:
# =========================
# VISUALIZE RESULTS
# =========================

model.eval()

with torch.no_grad():
    noisy, clean = next(iter(test_loader))
    noisy = noisy.to(device)
    clean = clean.to(device)

    output = model(noisy)

# Convert tensors to numpy
noisy_np = noisy.squeeze(0).permute(1,2,0).cpu().numpy()
clean_np = clean.squeeze(0).permute(1,2,0).cpu().numpy()
out_np = output.squeeze(0).permute(1,2,0).cpu().numpy()

# Clip for display
noisy_np = np.clip(noisy_np, 0, 1)
clean_np = np.clip(clean_np, 0, 1)
out_np = np.clip(out_np, 0, 1)

import matplotlib.pyplot as plt

plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.title("Low spp Noisy render")
plt.imshow(noisy_np)
plt.axis("off")

plt.subplot(1,3,2)
plt.title("Model Output")
plt.imshow(out_np)
plt.axis("off")

plt.subplot(1,3,3)
plt.title("High spp Reference")
plt.imshow(clean_np)
plt.axis("off")

plt.show()

In [ ]:
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

# for i in range(100):
#     optimizer.zero_grad()
#     output = model(noisy)
#     loss = torch.nn.functional.l1_loss(output, clean)
#     loss.backward()
#     optimizer.step()

#     if i % 20 == 0:
#         print(i, loss.item())